<a href="https://colab.research.google.com/github/Berubu/IA-Alcaraz/blob/main/P4Tutor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import torch
major_version, minor_version = torch.cuda.get_device_capability()
# Colab requiere una instalación especial para evitar errores de compatibilidad
if major_version >= 8:
    !pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
else:
    !pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Instalamos las dependencias mínimas necesarias
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-s_74mfgm/unsloth_08823403a7fa4ee0890d85a1e620e95d
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-s_74mfgm/unsloth_08823403a7fa4ee0890d85a1e620e95d
  Resolved https://github.com/unslothai/unsloth.git to commit d5b61a6bc6d546ca6afa9043f9c4b15713ecac62
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [22]:
# Instalamos unsloth_zoo que es lo que te pide el error
!pip install --no-deps unsloth_zoo
# Actualizamos trl a una versión compatible con el "zoo"
!pip install trl==0.12.0

In [23]:
from unsloth import FastLanguageModel
import torch

# 1. Configuración del motor Llama-3
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 2. Configuración LoRA técnica del Tutor
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

print("Modelo Llama-3 listo para estudiar tus algoritmos.")

==((====))==  Unsloth 2025.12.7: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Modelo Llama-3 listo para estudiar tus algoritmos.


In [24]:
import os
archivo = "dataset_algoritmos.jsonl"
if os.path.exists(archivo):
    print(f"Archivo '{archivo}' encontrado. ¡Puedes continuar!")
else:
    print(f"El archivo '{archivo}' NO está en /content/. Por favor, súbelo de nuevo.")

Archivo 'dataset_algoritmos.jsonl' encontrado. ¡Puedes continuar!


In [25]:
from unsloth.chat_templates import get_chat_template

# Aplicamos el formato oficial de Llama-3 al tokenizer
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
)

def format_prompt(examples):
    texts = []
    for messages in examples["messages"]:
        # Ahora el tokenizer ya sabe usar apply_chat_template
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return { "text" : texts }

# Ahora sí, mapeamos el dataset
dataset = dataset.map(format_prompt, batched=True)
print(f"Dataset procesado correctamente. Total de ejemplos: {len(dataset)}")

Map:   0%|          | 0/1400 [00:00<?, ? examples/s]

Dataset procesado correctamente. Total de ejemplos: 1400


In [26]:
from datasets import load_dataset

# Usamos ambos archivos para que el tutor aprenda de todo el material
data_files = {
    "train": ["dataset_algoritmos.jsonl", "dataset_algoritmos_basico_400.jsonl"]
}

print("Cargando datos pedagógicos...")
dataset = load_dataset("json", data_files=data_files, split="train")

def format_prompt(examples):
    texts = []
    for messages in examples["messages"]:
        # Aplicamos el formato de chat de Llama-3
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(format_prompt, batched=True)
print(f"Dataset listo. El tutor tiene {len(dataset)} ejemplos para estudiar.")

Cargando datos pedagógicos...


Map:   0%|          | 0/1400 [00:00<?, ? examples/s]

Dataset listo. El tutor tiene 1400 ejemplos para estudiar.


In [27]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        num_train_epochs = 3,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 300, # Puedes subirlo a 120 si quieres mayor precisión pedagógica
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print(" Iniciando entrenamiento...")
trainer_stats = trainer.train()

# --- GUARDAR EL TRABAJO ---
# Guardamos los nuevos archivos (el adaptador de Llama-3)
model.save_pretrained("tutor_llama3_final")
tokenizer.save_pretrained("tutor_llama3_final")
print("¡Entrenamiento completado y tutor guardado'!")

/content/unsloth_compiled_cache/UnslothSFTTrainer.py:747: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/content/unsloth_compiled_cache/UnslothSFTTrainer.py:761: UserWarning: You passed a `dataset_num_proc` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/content/unsloth_compiled_cache/UnslothSFTTrainer.py:775: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map (num_proc=2):   0%|          | 0/1400 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


 Iniciando entrenamiento...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,400 | Num Epochs = 3 | Total steps = 400
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 83,886,080 of 8,114,147,328 (1.03% trained)


wandb: WARNING Failed to wrap stdout. Console logs will not be captured.
wandb: WARNING Failed to wrap stderr. Console logs will not be captured.


Step,Training Loss
1,2.846200
2,2.919200
3,2.587100
4,2.172900
5,1.737000
6,1.600300
7,1.752700
8,1.091300
9,1.154000
10,0.781700


train/epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇██
train/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
train/grad_norm,█ ▅▄▂▂▁▁▂▂▂▂▁▁▁▂▂▁▂▁▁▁▂▂▂▃▃▁▂▁▂▁▁▂▂▁▁▂▂▁
train/learning_rate,████▇▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▁▁▁▁
train/loss,▂▂▄▅██▁▁▂█▃▁▁▃▃▅▁▃▁▂▁▂▂▁▃▂▁▁▁▁▁▂▁▁▁▁▁▁▁▁
total_flos,2.648692261527552e+16
train/epoch,2.28571
train/global_step,400
train/grad_norm,0.08915
train/learning_rate,0.0
train/loss,0.3224


¡Entrenamiento completado y tutor guardado'!


In [31]:
def evaluar_tutor(pregunta):
    # 1. Usamos el template oficial para que no "se escape" el texto
    messages = [
        {"role": "user", "content": pregunta},
    ]

    # add_generation_prompt=True le dice: "ahora te toca hablar a ti como assistant"
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    # 2. Generación con "Freno de Mano" (Stop Tokens)
    outputs = model.generate(
        input_ids = inputs,
        max_new_tokens = 500,
        temperature = 0.1,
        repetition_penalty = 1.2,
        # Forzamos a que se detenga si ve el fin de turno de Llama-3
        eos_token_id = tokenizer.eos_token_id,
        pad_token_id = tokenizer.pad_token_id
    )

    # 3. Limpieza: solo devolvemos lo que escribió el asistente
    decoded = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)

    # Limpieza extra por si acaso se filtra la palabra 'user' o 'assistant'
    respuesta_limpia = decoded.split("user")[0].split("assistant")[0].strip()
    return respuesta_limpia

# Prueba ahora con un tema de complejidad
print(evaluar_tutor("¿Qué es la complejidad temporal y por qué es importante en algoritmos?"))

La complejidad temporal es el concepto básico de "cuánto tiempo tarda un programa en ejecutarse".

**Ejemplo:**
- **O(1) Constante:** Un bucle infinito. Si lo detienes a los 5 segundos, todavía tardas igual.
- **O(n) Lineal:** Recorrer una lista de contactos. Cuanto más largo sea la lista, más rápido se demora.

**Importancia:**
Evitar programas lentísimos o inacabables. A menudo se combina con otras complejidades (espacial, memoria...).


In [1]:
import shutil

# Comprimimos la carpeta del nuevo modelo
shutil.make_archive("tutor_llama3_entrenado", 'zip', "tutor_llama3_final")

# Descargamos el archivo a tu PC
from google.colab import files
files.download("tutor_llama3_entrenado.zip")

FileNotFoundError: [Errno 2] No such file or directory: 'tutor_llama3_final'